# 01 · Video → frames

`football_eval/video.py::iter_frames` decodes the mp4 and keeps every `step = round(fps / analysis_fps)`-th frame, on
**one frame grid** anchored at the start of the first window (so both halves share the grid). Frames that are
skipped are only `grab()`-ed (no decoding to BGR), and decoding runs in a background thread (`prefetch`) so it
overlaps with the detector.

Then, before any model: the **grass mask** (`people.grass_top`) finds the far edge of the pitch in each column; the
detector only looks below it (minus some head room) when there is no calibration yet (`people.grass_roi`).

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

## 1 · The frame grid

In [ ]:
from football_eval.video import iter_frames, read_frame, fmt_clock
rows = [(f, t) for f, t, _ in nbt.frames_in_window(ctx, prefetch=0)]
idx = pd.DataFrame(rows, columns=["frame", "t"])
d = np.diff(idx.frame)
print(f"{len(idx)} analysed frames, frame step {set(d.tolist())}, first {idx.frame.iloc[0]} ({fmt_clock(idx.t.iloc[0])}), "
      f"last {idx.frame.iloc[-1]} ({fmt_clock(idx.t.iloc[-1])})")
assert (d == ctx.step).all(), "the grid should be regular inside one window"
idx.head()

## 2 · Decoding speed
Sequential decoding with `grab()` for skipped frames vs seeking every frame (`read_frame`), and the effect of the prefetch thread. On a full match decoding is ~5 % of the time; seeking is much slower — never seek per frame.

In [ ]:
def bench(n=60, **kw):
    t0 = time.perf_counter()
    k = sum(1 for _ in nbt.frames_in_window(ctx, max_frames=n, **kw))
    return k / (time.perf_counter() - t0)
seq = bench(prefetch=0); pre = bench(prefetch=8)
t0 = time.perf_counter()
for t in idx.t.iloc[:15]:
    read_frame(ctx.video, float(t))
seek = 15 / (time.perf_counter() - t0)
pd.DataFrame({"frames / s": [seq, pre, seek]}, index=["sequential (grab skipped)", "sequential + prefetch thread", "seek every frame"]).round(1)

## 3 · What the frames look like across the window

In [ ]:
imgs, titles = [], []
pick = set(np.linspace(0, len(idx) - 1, 6).round().astype(int).tolist())
for k, (f, t, img) in enumerate(nbt.frames_in_window(ctx)):
    if k in pick:
        imgs.append(img); titles.append(f"frame {f}  t={fmt_clock(t)}")
nbt.grid(imgs, titles, cols=3, size=5.5);

## 4 · Grass mask and the detector region (before any calibration)
`grass_top` = far edge of the grass per column (yellow). `grass_roi` = rows the detector will look at (magenta) when no homography is available. With a calibration, the integrated loop uses the *previous frame's* pitch outline instead (`perception.pitch_roi`, see 03).

In [ ]:
from football_eval.people import grass_top, grass_roi
f, t, img = nbt.probe_frame(ctx)
gt = grass_top(img)
roi = grass_roi(gt, img.shape)
vis = img.copy()
pts = np.c_[np.arange(len(gt)), gt].astype(np.int32)
cv2.polylines(vis, [pts.reshape(-1, 1, 2)], False, (0, 255, 255), 3)
if roi is not None:
    cv2.rectangle(vis, (int(roi[0]), int(roi[1])), (int(roi[2]) - 1, int(roi[3]) - 1), (255, 0, 255), 4)
print("grass ROI (x0, y0, x1, y1):", roi, f"-> skips the top {roi[1] / img.shape[0]:.0%} of the frame" if roi else "")
nbt.show(vis, "far edge of the grass (yellow) and detector ROI (magenta)");

## 5 · Camera motion over the window
Mean absolute difference between consecutive analysed frames (downscaled): peaks = pans / zooms. The calibration (03) and the ball tracker (05) must follow these.

In [ ]:
prev, md = None, []
for f, t, img in nbt.frames_in_window(ctx):
    g = cv2.cvtColor(cv2.resize(img, (320, 180)), cv2.COLOR_BGR2GRAY).astype(np.float32)
    md.append((f, t, np.nan if prev is None else float(np.abs(g - prev).mean())))
    prev = g
mdf = pd.DataFrame(md, columns=["frame", "t", "mean_abs_diff"])
ax = mdf.plot(x="t", y="mean_abs_diff", figsize=(11, 2.8), legend=False)
ax.set_xlabel("video time (s)"); ax.set_ylabel("mean |Δ| (grey levels)"); ax.set_title("frame-to-frame change (camera motion + players)");

In [ ]:
nbt.save_stage(ctx, "frames_index", idx.merge(mdf, on=["frame", "t"]));